# Grammar Scoring Engine for Spoken Audio (SHL Hiring Assessment 2026)

Predict a continuous MOS grammar score (0-5) from a 45-60 s speech clip. Metrics: **RMSE** and **Pearson correlation**.

## Report

**Approach.** Grammar is a property of *what* is said, so audio is first converted to text with Whisper (ASR) and the text is scored. Acoustic and fluency features (pauses, speech ratio, MFCC statistics) and a wav2vec2 audio embedding are added, because ASR partly "repairs" grammar and because hesitations correlate with grammar level.

**Pipeline**
1. Load audio (16 kHz), check durations.
2. ASR transcripts with Whisper (cached, resumable).
3. Feature blocks: (a) acoustic + fluency, (b) hand-crafted text features incl. GPT-2 perplexity, (c) RoBERTa transcript embedding, (d) wav2vec2 audio embedding.
4. Models: ExtraTrees, XGBoost and LightGBM on (a)+(b); Ridge on (c) and on (d).
5. 5-fold CV out-of-fold (OOF) predictions, then a non-negative linear stack, clipped to [0, 5].

**Evaluation.** Training RMSE (fit on all training data, required) and 5-fold OOF RMSE / Pearson (honest estimate of generalisation).

**Note on the test set.** 212 of the 216 test *filenames* also appear in `train.csv`, but they are **different recordings** (same name, different audio, in the `test/` folder). Looking up the training label by filename therefore gives no signal (a leaderboard RMSE of ~1.25, about the same as predicting the mean). Train and test audio are kept strictly separate (keys `train/<file>` and `test/<file>`), and every test prediction is made by the model from the test audio itself.

## 1. Setup

In [ ]:
# Run once if a package is missing:
# !pip install -q xgboost lightgbm
import os, re, gc, warnings, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch, librosa, soundfile as sf
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from scipy.optimize import nnls
from sklearn.model_selection import KFold
from sklearn.base import clone
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import ExtraTreesRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SR = 16000
DATA_PATH = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
WORK = "/kaggle/working"
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU. ASR + embeddings will take hours. In Kaggle: Settings > Accelerator > GPU T4 x2, then re-run.")

## 2. Data loading and EDA

In [ ]:
train = pd.read_csv(f"{DATA_PATH}/train.csv"); test = pd.read_csv(f"{DATA_PATH}/test.csv")
sample_sub = pd.read_csv(f"{DATA_PATH}/sample_submission.csv")
FILE_COL, LABEL_COL = "filename", "label"
SUB_LABEL_COL = [c for c in sample_sub.columns if c != FILE_COL][0]

# IMPORTANT: train and test contain files with the SAME NAME but DIFFERENT audio.
# Never key audio by bare file name; use "train/<file>" and "test/<file>".
for d in ("train", "test"): assert os.path.isdir(f"{DATA_PATH}/{d}"), f"missing folder {DATA_PATH}/{d}"
keys = [f"train/{f}" for f in train[FILE_COL]] + [f"test/{f}" for f in test[FILE_COL]]
path_of = {k: os.path.join(DATA_PATH, k) for k in keys}
N_TRAIN, N_TEST = len(train), len(test); y = train[LABEL_COL].to_numpy(float)
missing_audio = [k for k in keys if not os.path.exists(path_of[k])]
assert not missing_audio, f"missing audio: {missing_audio[:5]}"
print(f"train={N_TRAIN} | test={N_TEST} | test filenames also in train.csv={len(set(test[FILE_COL]) & set(train[FILE_COL]))}")

In [ ]:
# Are the same-named train/test files the same audio? (compare file hashes)
import hashlib
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
common = sorted(set(train[FILE_COL]) & set(test[FILE_COL]))[:30]
same = [md5(path_of[f"train/{f}"]) == md5(path_of[f"test/{f}"]) for f in common]
print(f"{sum(same)} of {len(common)} same-named train/test files are identical audio")

In [ ]:
dur = pd.Series({k: sf.info(path_of[k]).duration for k in keys}, name="duration")
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
train[LABEL_COL].value_counts().sort_index().plot.bar(ax=ax[0], title="Label distribution"); ax[0].set_xlabel("Grammar score")
ax[1].hist(dur, bins=30); ax[1].set(title="Audio duration (s)", xlabel="seconds")
ax[2].scatter(dur.iloc[:N_TRAIN].values, y, alpha=.4); ax[2].set(title="Duration vs score", xlabel="seconds", ylabel="score")
plt.tight_layout(); plt.show()
print("Baseline RMSE (predict the mean):", round(float(np.sqrt(np.mean((y - y.mean())**2))), 4))

## 3. ASR transcription (Whisper)
Cached to `/kaggle/working/transcripts_all.csv` and resumable: if the session stops, re-run the cell and it continues where it left off.

In [ ]:
from transformers import pipeline
ASR_MODEL = "openai/whisper-small.en" if DEVICE == "cuda" else "openai/whisper-base.en"
TRANS_CSV = f"{WORK}/transcripts_v3.csv"
OLD_TRANS = f"{WORK}/transcripts_all.csv"   # earlier cache keyed by bare file name

def transcribe(files):
    done = pd.read_csv(TRANS_CSV).fillna("") if os.path.exists(TRANS_CSV) else pd.DataFrame(columns=["filename", "transcript"])
    have = dict(zip(done.filename, done.transcript))
    if not have and os.path.exists(OLD_TRANS):            # reuse old TRAIN transcripts only (test names collided with train)
        old = pd.read_csv(OLD_TRANS).fillna("")
        have = {f"train/{f}": t for f, t in zip(old.filename, old.transcript) if f in train_set_names}
        print("migrated", len(have), "old train transcripts")
    todo = [f for f in files if f not in have]
    print(f"ASR model {ASR_MODEL} | cached {len(have)} | to do {len(todo)}")
    if todo:
        asr = pipeline("automatic-speech-recognition", model=ASR_MODEL, device=0 if DEVICE == "cuda" else -1)
        for i, f in enumerate(tqdm(todo, desc="ASR")):
            try:
                have[f] = asr(path_of[f], return_timestamps=True)["text"].strip()
            except Exception as e:
                print("ASR error", f, e); have[f] = ""
            if (i + 1) % 25 == 0:
                pd.DataFrame({"filename": list(have), "transcript": list(have.values())}).to_csv(TRANS_CSV, index=False)
        pd.DataFrame({"filename": list(have), "transcript": list(have.values())}).to_csv(TRANS_CSV, index=False)
        del asr; gc.collect()
    return [have[f] for f in files]

train_set_names = set(train[FILE_COL])
texts = transcribe(keys)
for i in range(3): print(f"[score={y[i]}] {texts[i][:250]}\n")

## 4. Feature engineering

### 4.1 Acoustic and fluency features

In [ ]:
def acoustic_features(path):
    wav, _ = librosa.load(path, sr=SR)
    f = {}
    mfcc = librosa.feature.mfcc(y=wav, sr=SR, n_mfcc=13)
    for i in range(13):
        f[f"mfcc{i}_mean"] = mfcc[i].mean(); f[f"mfcc{i}_std"] = mfcc[i].std()
    f["zcr"] = librosa.feature.zero_crossing_rate(wav).mean()
    f["centroid"] = librosa.feature.spectral_centroid(y=wav, sr=SR).mean()
    f["bandwidth"] = librosa.feature.spectral_bandwidth(y=wav, sr=SR).mean()
    f["rolloff"] = librosa.feature.spectral_rolloff(y=wav, sr=SR).mean()
    f["rms"] = librosa.feature.rms(y=wav).mean()
    f["duration"] = len(wav) / SR
    # pause / fluency features
    iv = librosa.effects.split(wav, top_db=30)
    gaps = (iv[1:, 0] - iv[:-1, 1]) / SR if len(iv) > 1 else np.array([0.0])
    f["speech_ratio"] = (iv[:, 1] - iv[:, 0]).sum() / max(len(wav), 1) if len(iv) else 0.0
    f["n_segments"] = len(iv); f["mean_gap"] = gaps.mean(); f["n_long_pauses"] = int((gaps > 0.5).sum())
    return f

AC_CSV = f"{WORK}/acoustic_v3.csv"
if os.path.exists(AC_CSV): ac_df = pd.read_csv(AC_CSV)
else:
    ac_df = pd.DataFrame([acoustic_features(path_of[k]) for k in tqdm(keys, desc="acoustic")]); ac_df.to_csv(AC_CSV, index=False)
print(ac_df.shape)

### 4.2 Text features (including GPT-2 perplexity)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
lm_tok = AutoTokenizer.from_pretrained("distilgpt2")
lm = AutoModelForCausalLM.from_pretrained("distilgpt2").to(DEVICE).eval()

@torch.no_grad()
def log_ppl(text):
    ids = lm_tok(text, return_tensors="pt", truncation=True, max_length=512).input_ids.to(DEVICE)
    return float(lm(ids, labels=ids).loss.item()) if ids.shape[1] > 1 else np.nan

FILLERS = {"um", "uh", "hmm", "like", "actually", "basically", "know"}
PRON = {"i", "we", "you", "he", "she", "they", "it", "me", "us", "them"}
CONJ = {"and", "but", "or", "because", "although", "while", "if", "so", "when"}

def text_features(text, duration):
    words = re.findall(r"[a-zA-Z']+", text.lower())
    n = len(words)
    sents = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
    if n == 0:
        return {"n_words": 0, "wps": 0, "ttr": 0, "avg_wlen": 0, "long_word_rate": 0, "n_sents": 0, "avg_sent_len": 0,
                "max_sent_len": 0, "filler_rate": 0, "pron_rate": 0, "conj_rate": 0, "repeat_rate": 0, "bigram_repeat": 0, "log_ppl": np.nan}
    uniq = set(words)
    return {
        "n_words": n, "wps": n / max(duration, 1), "ttr": len(uniq) / n,
        "avg_wlen": np.mean([len(w) for w in words]), "long_word_rate": np.mean([len(w) > 6 for w in words]),
        "n_sents": len(sents), "avg_sent_len": n / max(len(sents), 1), "max_sent_len": max(len(s.split()) for s in sents),
        "filler_rate": sum(w in FILLERS for w in words) / n, "pron_rate": sum(w in PRON for w in words) / n,
        "conj_rate": sum(w in CONJ for w in words) / n,
        "repeat_rate": sum(a == b for a, b in zip(words, words[1:])) / n,
        "bigram_repeat": 1 - len(set(zip(words, words[1:]))) / max(n - 1, 1),
        "log_ppl": log_ppl(text),
    }

tx_df = pd.DataFrame([text_features(t, d) for t, d in tqdm(zip(texts, dur[keys].values), total=len(texts), desc="text feats")])
tx_df = tx_df.fillna(tx_df.median())
del lm; gc.collect()
hand_df = pd.concat([ac_df, tx_df.add_prefix("t_")], axis=1)
print("hand-crafted feature matrix:", hand_df.shape)

### 4.3 Transformer embeddings (text and audio)

In [ ]:
@torch.no_grad()
def text_embed(texts, name="roberta-base", bs=16):
    tok = AutoTokenizer.from_pretrained(name); m = AutoModel.from_pretrained(name).to(DEVICE).eval()
    out = []
    for i in tqdm(range(0, len(texts), bs), desc="text-emb"):
        b = tok([t if t.strip() else "empty" for t in texts[i:i+bs]], padding=True, truncation=True, max_length=512, return_tensors="pt").to(DEVICE)
        h = m(**b).last_hidden_state; mask = b["attention_mask"].unsqueeze(-1)
        out.append(((h * mask).sum(1) / mask.sum(1)).cpu().numpy())
    del m; gc.collect(); return np.vstack(out)

@torch.no_grad()
def audio_embed(files, name="facebook/wav2vec2-base-960h", layers=(6, 9, 12)):
    from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model
    fe = Wav2Vec2FeatureExtractor.from_pretrained(name); m = Wav2Vec2Model.from_pretrained(name).to(DEVICE).eval()
    out = []
    for f in tqdm(files, desc="audio-emb"):
        wav, _ = librosa.load(path_of[f], sr=SR)
        x = fe(wav, sampling_rate=SR, return_tensors="pt").input_values.to(DEVICE)
        hs = m(x, output_hidden_states=True).hidden_states
        out.append(np.concatenate([hs[l].mean(1).squeeze(0).cpu().numpy() for l in layers]))
    del m; gc.collect(); return np.vstack(out)

def cached(fn, name, *a):
    p = f"{WORK}/{name}.npy"
    if os.path.exists(p): return np.load(p)
    x = fn(*a); np.save(p, x); return x

T_all = cached(text_embed, "text_emb_v3", texts)
A_all = cached(audio_embed, "audio_emb_v3", keys)
print("text emb:", T_all.shape, "| audio emb:", A_all.shape)

## 5. Modelling
Tree models use the hand-crafted block; Ridge models use each embedding. All are evaluated with the same 5-fold split. Test predictions are the average of the five fold models, computed from the test audio.

In [ ]:
H_all = hand_df.to_numpy(float)
rmse = lambda a, b: float(np.sqrt(mean_squared_error(a, b)))
clip = lambda p: np.clip(p, 0, 5)

specs = {
    "ExtraTrees": (H_all, ExtraTreesRegressor(n_estimators=500, max_depth=10, min_samples_leaf=2, random_state=SEED, n_jobs=-1)),
    "XGBoost":    (H_all, XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1)),
    "LightGBM":   (H_all, LGBMRegressor(n_estimators=500, learning_rate=0.03, max_depth=4, num_leaves=15, subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbosity=-1)),
    "Ridge_text_emb":  (T_all, make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(1, 5, 20)))),
    "Ridge_audio_emb": (A_all, make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(1, 5, 20)))),
    "Ridge_all": (np.hstack([H_all, T_all, A_all]), make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(2, 6, 20)))),
}

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof, new_pred, train_pred, rows = {}, {}, {}, []
for name, (X, mdl) in specs.items():
    Xtr, Xnew = X[:N_TRAIN], X[N_TRAIN:]   # train rows, then test rows
    o, nw = np.zeros(N_TRAIN), np.zeros(len(Xnew))
    for tr, va in kf.split(Xtr):
        m = clone(mdl).fit(Xtr[tr], y[tr])
        o[va] = m.predict(Xtr[va]); nw += m.predict(Xnew) / kf.get_n_splits() if len(Xnew) else 0
    full = clone(mdl).fit(Xtr, y)
    oof[name], new_pred[name], train_pred[name] = clip(o), clip(nw), clip(full.predict(Xtr))
    rows.append({"model": name, "OOF_RMSE": rmse(y, oof[name]), "OOF_Pearson": pearsonr(y, oof[name])[0], "Train_RMSE": rmse(y, train_pred[name])})
res = pd.DataFrame(rows).sort_values("OOF_RMSE"); display(res.round(4))

### Stacking

In [ ]:
names = list(specs)
P_oof = np.column_stack([oof[n] for n in names]); P_tr = np.column_stack([train_pred[n] for n in names])
P_new = np.column_stack([new_pred[n] for n in names])
w, _ = nnls(P_oof, y); w = w / w.sum()
print(pd.Series(w, index=names).round(3).to_string())

stack_oof, stack_train, stack_test = clip(P_oof @ w), clip(P_tr @ w), clip(P_new @ w)
TRAIN_RMSE = rmse(y, stack_train)
print("\n" + "=" * 58)
print(f"FINAL TRAINING RMSE (fit on all train): {TRAIN_RMSE:.4f} | Train Pearson: {pearsonr(y, stack_train)[0]:.4f}")
print(f"5-fold OOF RMSE: {rmse(y, stack_oof):.4f} | OOF Pearson: {pearsonr(y, stack_oof)[0]:.4f}")
print("=" * 58)

## 6. Evaluation and visualisation

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ax[0].scatter(y, stack_oof, alpha=.4); ax[0].plot([0, 5], [0, 5], "r--")
ax[0].set(xlabel="True score", ylabel="Predicted (OOF)", title=f"OOF predicted vs true (r={pearsonr(y, stack_oof)[0]:.3f})")
sns.histplot(y - stack_oof, kde=True, ax=ax[1]); ax[1].set_title("OOF residuals")
res.set_index("model")[["OOF_RMSE", "Train_RMSE"]].plot.barh(ax=ax[2], title="RMSE by model"); plt.tight_layout(); plt.show()

err = pd.DataFrame({"y": y, "abs_err": np.abs(y - stack_oof)}).groupby("y").abs_err.mean()
err.plot.bar(figsize=(7, 3.5), title="Mean |error| by true score"); plt.show()

In [ ]:
# Interpretability: which hand-crafted features matter and how they correlate with the score
et = clone(specs["ExtraTrees"][1]).fit(H_all[:N_TRAIN], y)
imp = pd.Series(et.feature_importances_, index=hand_df.columns).sort_values().tail(15)
corr = hand_df.iloc[:N_TRAIN].assign(score=y).corr()["score"].drop("score")
corr = corr.reindex(corr.abs().sort_values().tail(15).index).sort_values()
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
imp.plot.barh(ax=ax[0], title="ExtraTrees top-15 feature importance"); corr.plot.barh(ax=ax[1], title="Top-15 |correlation| with score")
plt.tight_layout(); plt.show()

# Low-scored (<=1) clips: are they empty / unintelligible?
low = train.assign(text=texts[:N_TRAIN], n_words=tx_df["n_words"].values[:N_TRAIN])
print("mean words | score 0:", low.loc[low.label == 0, "n_words"].mean().round(1), "| score>=3:", low.loc[low.label >= 3, "n_words"].mean().round(1))

## 7. Test predictions and submission
One row per `test.csv` file (216), predicted by the model from the **test** audio.

In [ ]:
submission = test[[FILE_COL]].copy()
submission[SUB_LABEL_COL] = clip(stack_test).astype(float)
assert len(submission) == len(test) and submission[SUB_LABEL_COL].notna().all()
submission.to_csv(f"{WORK}/submission.csv", index=False)
print("Saved:", submission.shape, "| range:", round(float(submission[SUB_LABEL_COL].min()), 2), "-", round(float(submission[SUB_LABEL_COL].max()), 2),
      "| unique values:", submission[SUB_LABEL_COL].nunique())
display(submission.head())
sns.histplot(submission[SUB_LABEL_COL], bins=20); plt.title("Distribution of test predictions"); plt.show()

## 8. Discussion
- **What drives the score:** speech volume and fluency (words per second, pauses, speech ratio), lexical diversity, language-model perplexity of the transcript, and learned embeddings. Very low scores (0) tend to be near-empty or unintelligible clips, which the text and pause features separate easily.
- **Limitations:** Whisper partly corrects grammar mistakes, weakening text features; labels are noisy ordinal ratings; only 769 training clips.
- **Improvements:** larger Whisper model, ASR word-confidence features, LanguageTool error rate (needs Java), fine-tuning a speech encoder with a regression head, repeated-seed CV ensembles.